In [2]:
import cv2
import numpy as np

# 1. Initialize Video
cap = cv2.VideoCapture('Input_video\match_01.mp4')

if not cap.isOpened():
    print("Error: Cannot open video")
    exit()

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 30

# Better codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    'footballplayer_tracked_box.mp4',
    fourcc,
    fps,
    (width, height)
)

# 2. Background Subtractor
backSub = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=100,
    detectShadows=False
)

print("Processing: Tracking the football player...")

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # ROI Mask
    mask_roi = np.zeros(frame.shape[:2], dtype="uint8")

    cv2.rectangle(
        mask_roi,
        (0, int(height * 0.2)),
        (width, int(height * 0.8)),
        255,
        -1
    )

    # Background subtraction
    fg_mask = backSub.apply(frame)

    fg_mask = cv2.bitwise_and(
        fg_mask,
        fg_mask,
        mask=mask_roi
    )

    # Morphological operations
    kernel = np.ones((15, 15), np.uint8)

    fg_mask = cv2.morphologyEx(
        fg_mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    fg_mask = cv2.dilate(
        fg_mask,
        kernel,
        iterations=1
    )

    # Find contours
    contours, _ = cv2.findContours(
        fg_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:

        c = max(contours, key=cv2.contourArea)

        if cv2.contourArea(c) > 1500:

            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                3
            )

            cv2.putText(
                frame,
                "TRACKING",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    out.write(frame)

# Release
cap.release()
out.release()

print("Done!")

<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:5: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Lap Mate\AppData\Local\Temp\ipykernel_7188\2355895555.py:5: SyntaxWarning: invalid escape sequence '\m'
  cap = cv2.VideoCapture('Input_video\match_01.mp4')


Processing: Tracking the football player...
Done!
